# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 10.3 MB/s eta 0:00:00
dependencies ok


In [4]:
import json, os, time, hashlib, zipfile,  csv, base64, pickle
import glob, sys,math, random, collections,io,shutil
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict, Counter
from onnx import shape_inference

In [5]:
torch.set_num_threads(1)

In [6]:
TASK_ID = "task328"
CH = 10
H = W = 30
LOCAL_TASK_JSON = Path("/mnt/data/task328(1).json")
KAGGLE_TASK_JSON = Path(COMPETITION) / "task328.json"
TASK_JSON = LOCAL_TASK_JSON if LOCAL_TASK_JSON.exists() else KAGGLE_TASK_JSON
OUT_DIR = Path.cwd() / "task328_alt_model_onnx"
OUT_DIR.mkdir(parents=True, exist_ok=True)
ONNX_PATH = OUT_DIR / f"{TASK_ID}.onnx"
SUBMISSION_PATH = Path.cwd() / "submission.zip"
SUMMARY_PATH = OUT_DIR / f"{TASK_ID}_alt_validation_summary.json"
with TASK_JSON.open("r") as f:
    task = json.load(f)
print(TASK_ID, len(task.get("train", [])), len(task.get("test", [])), len(task.get("arc-gen", [])))

task328 4 1 262


In [7]:

def grid_to_tensor(grid, h=H, w=W, ch=CH, full_background=False):
    x=np.zeros((1,ch,h,w),dtype=np.float32)
    if full_background:
        x[0,0,:,:]=1.0
    for r,row in enumerate(grid):
        for c,v in enumerate(row):
            if full_background:
                x[0,:,r,c]=0.0
            x[0,int(v),r,c]=1.0
    return x

def compose_output(canvas, masks):
    outs=[]; occ=torch.zeros_like(canvas)
    for k in range(1,10):
        m=masks.get(k, torch.zeros_like(canvas))
        m=(m>0.5).float()*canvas
        outs.append(m)
        occ=torch.clamp(occ+m,0,1)
    return torch.cat([canvas*(1.0-torch.clamp(occ,0,1))]+outs,dim=1)*canvas
class Task328HybridCorners(nn.Module):
    def __init__(self):
        super().__init__()
        rr=torch.arange(30,dtype=torch.float32).view(1,1,30,1).expand(1,1,30,30)
        cc=torch.arange(30,dtype=torch.float32).view(1,1,1,30).expand(1,1,30,30)
        self.register_buffer('rr',rr); self.register_buffer('cc',cc)
    def base_cond(self, rr, cc):
        r_even=(torch.floor(rr/2.0)*2.0==rr).float()
        c_even=(torch.floor(cc/2.0)*2.0==cc).float()
        return torch.clamp(r_even*(cc<=rr).float()+c_even*(rr<=cc).float(),0,1)
    def forward(self,x):
        raw_active=(x.sum(dim=1,keepdim=True)>0.5).float()
        nz=(x[:,1:10].sum(dim=1,keepdim=True)>0.5).float()
        row_act=(raw_active.sum(dim=3,keepdim=True)>0.5).float()
        col_act=(raw_active.sum(dim=2,keepdim=True)>0.5).float()
        h_act=(row_act*self.rr[:,:,:,0:1]).amax(dim=2,keepdim=True)
        w_act=(col_act*self.cc[:,:,0:1,:]).amax(dim=3,keepdim=True)
        row_nz=(nz.sum(dim=3,keepdim=True)>0.5).float()
        col_nz=(nz.sum(dim=2,keepdim=True)>0.5).float()
        h_nz=(row_nz*self.rr[:,:,:,0:1]).amax(dim=2,keepdim=True)
        w_nz=(col_nz*self.cc[:,:,0:1,:]).amax(dim=3,keepdim=True)
        # Hybrid size: when a right/bottom corner color is present use nonzero extrema; otherwise use true active canvas.
        hmax=torch.where(h_nz>0.5, torch.minimum(h_act,h_nz), h_act)
        wmax=torch.where(w_nz>0.5, torch.minimum(w_act,w_nz), w_act)
        canvas=(self.rr<=hmax).float()*(self.cc<=wmax).float()*raw_active
        # if raw_active is accidentally full 30x30, still restrict to inferred nonzero max if right/bottom corners exist
        canvas_fullsafe=(self.rr<=hmax).float()*(self.cc<=wmax).float()
        canvas=torch.where(raw_active.sum(dim=(2,3),keepdim=True)>800.0, canvas_fullsafe, canvas)
        right_col=(self.cc==wmax).float()*canvas
        bottom_row=(self.rr==hmax).float()*canvas
        tl_seed=x[:,:,0:1,0:1]
        tr_seed=(x*right_col)[:,:,0:1,:].sum(dim=3,keepdim=True)
        bl_seed=(x*bottom_row)[:,:,:,0:1].sum(dim=2,keepdim=True)
        br_seed=(x*bottom_row*right_col).sum(dim=(2,3),keepdim=True)
        tl_on=(tl_seed[:,1:10].sum(dim=1,keepdim=True)>0.5).float()
        tr_on=(tr_seed[:,1:10].sum(dim=1,keepdim=True)>0.5).float()
        bl_on=(bl_seed[:,1:10].sum(dim=1,keepdim=True)>0.5).float()
        br_on=(br_seed[:,1:10].sum(dim=1,keepdim=True)>0.5).float()
        rr=self.rr; cc=self.cc
        big=torch.tensor(1000.0,dtype=torch.float32,device=x.device)
        d_tl=rr+cc+(1.0-tl_on)*big
        d_tr=rr+(wmax-cc)+(1.0-tr_on)*big
        d_bl=(hmax-rr)+cc+(1.0-bl_on)*big
        d_br=(hmax-rr)+(wmax-cc)+(1.0-br_on)*big
        dmin=torch.minimum(torch.minimum(d_tl,d_tr),torch.minimum(d_bl,d_br))
        is_tl=(d_tl==dmin).float()*tl_on
        is_tr=(d_tr==dmin).float()*tr_on
        is_bl=(d_bl==dmin).float()*bl_on
        is_br=(d_br==dmin).float()*br_on
        single=(is_tl+is_tr+is_bl+is_br==1.0).float()
        cond_tl=self.base_cond(rr,cc)
        cond_tr=self.base_cond(rr,wmax-cc)
        cond_bl=self.base_cond(hmax-rr,cc)
        cond_br=self.base_cond(hmax-rr,wmax-cc)
        masks={}
        for k in range(1,10):
            masks[k]=canvas*single*torch.clamp(
                is_tl*cond_tl*tl_seed[:,k:k+1]+is_tr*cond_tr*tr_seed[:,k:k+1]+
                is_bl*cond_bl*bl_seed[:,k:k+1]+is_br*cond_br*br_seed[:,k:k+1],0,1)
        return compose_output(canvas,masks)

class Task345RayTracerNoScatter(nn.Module):
    def __init__(self):
        super().__init__()
        rr=torch.arange(30,dtype=torch.float32).view(1,1,30,1).expand(1,1,30,30)
        cc=torch.arange(30,dtype=torch.float32).view(1,1,1,30).expand(1,1,30,30)
        self.register_buffer('rr',rr); self.register_buffer('cc',cc)
    def shift_right(self,t):
        z=torch.zeros_like(t[:,:,:,:1])
        return torch.cat([z,t[:,:,:,:-1]],dim=3)
    def forward(self,x):
        canvas=(self.rr<10.0).float()*(self.cc<10.0).float()
        nz=x[:,1:10]*canvas
        bottom=(self.rr==9.0).float()*canvas
        bottom_counts=(nz*bottom).sum(dim=(2,3),keepdim=True)
        maxb=bottom_counts.amax(dim=1,keepdim=True)
        line_sel=(bottom_counts==maxb).float()*(maxb>0.5).float()
        line=(nz*line_sel).sum(dim=1,keepdim=True)
        blockers=torch.clamp(nz.sum(dim=1,keepdim=True)-line,0,1)*canvas
        cur=(line*bottom).sum(dim=2,keepdim=True).clamp(0,1)
        rows=[]
        for r in range(29,-1,-1):
            in_canvas_row=((self.rr[:,:,r:r+1,:]<10.0)&(self.cc[:,:,r:r+1,:]<10.0)).float()
            if r>0:
                obs=blockers[:,:,r-1:r,:]*cur
                shifted=self.shift_right(obs)
                bridge=shifted
                nxt=torch.clamp(cur*(1.0-obs)+shifted,0,1)
            else:
                bridge=torch.zeros_like(cur); nxt=cur
            row_line=torch.clamp(cur+bridge,0,1)*in_canvas_row*(1.0-blockers[:,:,r:r+1,:])
            rows.insert(0,row_line)
            cur=nxt
        fill=torch.cat(rows,dim=2)*canvas
        masks={}
        for idx,k in enumerate(range(1,10)):
            sel=line_sel[:,idx:idx+1]
            masks[k]=x[:,k:k+1]*canvas*(1.0-sel)+fill*sel
        return compose_output(canvas,masks)

class Task363ContextStampNoScatter(nn.Module):
    def __init__(self):
        super().__init__()
        rr30=torch.arange(30,dtype=torch.float32).view(1,1,30,1).expand(1,1,30,30)
        cc30=torch.arange(30,dtype=torch.float32).view(1,1,1,30).expand(1,1,30,30)
        rr59=torch.arange(59,dtype=torch.float32).view(1,1,59,1).expand(1,1,59,59)
        cc59=torch.arange(59,dtype=torch.float32).view(1,1,1,59).expand(1,1,59,59)
        self.register_buffer('rr30',rr30); self.register_buffer('cc30',cc30)
        self.register_buffer('rr59',rr59); self.register_buffer('cc59',cc59)
    def shift2d(self,t,dr,dc):
        # constant zero shift without ScatterND/Range
        out=t
        if dr>0:
            out=F.pad(out[:,:,:-dr,:],(0,0,dr,0))
        elif dr<0:
            out=F.pad(out[:,:,-dr:,:],(0,0,0,-dr))
        if dc>0:
            out=F.pad(out[:,:,:,:-dc],(dc,0,0,0))
        elif dc<0:
            out=F.pad(out[:,:,:,-dc:],(0,-dc,0,0))
        return out
    def forward(self,x):
        canvas=(self.rr30<10.0).float()*(self.cc30<10.0).float()
        nz=x[:,1:10]*canvas
        counts=nz.sum(dim=(2,3),keepdim=True)
        maxc=counts.amax(dim=1,keepdim=True)
        wall_sel=(counts==maxc).float()*(maxc>0.5).float()
        wall=(nz*wall_sel).sum(dim=1,keepdim=True)
        obj_sel=(1.0-wall_sel)*(counts>0.5).float()
        obj=(nz*obj_sel).sum(dim=1,keepdim=True)
        block=torch.clamp(wall+(1.0-canvas),0,1)
        valid=(F.conv2d(F.pad(block,(29,29,29,29),value=1.0),obj)<0.5).float()
        # constrain pure bars to same axis as the source bar
        row_presence=(obj.sum(dim=3,keepdim=True)>0.5).float()
        col_presence=(obj.sum(dim=2,keepdim=True)>0.5).float()
        row_n=row_presence.sum(dim=2,keepdim=True)
        col_n=col_presence.sum(dim=3,keepdim=True)
        is_hline=(row_n<1.5).float(); is_vline=(col_n<1.5).float()
        valid=valid*(is_hline*(self.rr59==29.0).float()+(1.0-is_hline))
        valid=valid*(is_vline*(self.cc59==29.0).float()+(1.0-is_vline))
        # context score: prefer placements whose object cells touch the same wall density; then non-overlap local maxima
        neigh=F.pad(wall,(1,1,1,1),value=1.0)
        nb=neigh[:,:,0:30,1:31]+neigh[:,:,2:32,1:31]+neigh[:,:,1:31,0:30]+neigh[:,:,1:31,2:32]
        score_raw=F.conv2d(F.pad(nb,(29,29,29,29),value=0.0),obj)
        score=valid*(score_raw+(self.rr59*0.0001-self.cc59*0.00001))
        local=score
        for dr in range(-2,3):
            for dc in range(-2,3):
                if dr==0 and dc==0: continue
                shifted_obj=self.shift2d(obj,dr,dc)
                overlap=(obj*shifted_obj).sum(dim=(2,3),keepdim=True)>0.5
                local=torch.maximum(local,self.shift2d(score,dr,dc)*overlap.float())
        selected=(score>=local-1e-6).float()*valid
        placed=F.conv_transpose2d(selected,obj)
        acc=(placed[:,:,29:59,29:59]>0.5).float()*canvas*(1.0-wall)
        masks={}
        for idx,k in enumerate(range(1,10)):
            masks[k]=wall*wall_sel[:,idx:idx+1]+acc*obj_sel[:,idx:idx+1]
        return compose_output(canvas,masks).reshape(1,10,30,30)


MODEL_CLASS = {328:Task328HybridCorners,345:Task345RayTracerNoScatter,363:Task363ContextStampNoScatter}[328]
model = MODEL_CLASS().eval()
print(model.__class__.__name__)

Task328HybridCorners


In [8]:
dummy = torch.from_numpy(grid_to_tensor(task["test"][0]["input"]))
torch.onnx.export(
    model, dummy, str(ONNX_PATH), input_names=["input"], output_names=["output"],
    opset_version=17, do_constant_folding=True, dynamic_axes=None, dynamo=False,
)
onnx_model = onnx.load(str(ONNX_PATH))
onnx_model = onnx.shape_inference.infer_shapes(onnx_model)
onnx.save(onnx_model, str(ONNX_PATH))
onnx.checker.check_model(str(ONNX_PATH))
ONNX_PATH, ONNX_PATH.stat().st_size

/tmp/ipykernel_16/1648413716.py:2: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/tmp/ipykernel_16/2375706853.py:59: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  big=torch.tensor(1000.0,dtype=torch.float32,device=x.device)


(PosixPath('/kaggle/working/task328_alt_model_onnx/task328.onnx'), 96441)

In [9]:
def vi_shape(vi):
    return [int(d.dim_value) if d.dim_value else (d.dim_param or None) for d in vi.type.tensor_type.shape.dim]
onnx_model = onnx.load(str(ONNX_PATH))
ops = collections.Counter(node.op_type for node in onnx_model.graph.node)
forbidden = {"Loop", "Scan", "NonZero", "Unique", "Script", "Function"}
summary_static = {
    "input_shape": vi_shape(onnx_model.graph.input[0]),
    "output_shape": vi_shape(onnx_model.graph.output[0]),
    "onnx_size_bytes": ONNX_PATH.stat().st_size,
    "ops": dict(ops),
    "forbidden_ops": sorted(forbidden & set(ops)),
}
print(summary_static)
assert summary_static["input_shape"] == [1,10,30,30]
assert summary_static["output_shape"] == [1,10,30,30]
assert summary_static["onnx_size_bytes"] < 1_400_000
assert not summary_static["forbidden_ops"]

{'input_shape': [1, 10, 30, 30], 'output_shape': [1, 10, 30, 30], 'onnx_size_bytes': 96441, 'ops': {'Constant': 269, 'ReduceSum': 14, 'Greater': 22, 'Cast': 40, 'Slice': 45, 'Mul': 92, 'ReduceMax': 4, 'Min': 5, 'Where': 3, 'LessOrEqual': 10, 'Equal': 11, 'Sub': 7, 'Add': 50, 'Floor': 4, 'Clip': 23, 'Div': 2, 'Concat': 1}, 'forbidden_ops': []}


In [10]:
sess_options = ort.SessionOptions(); sess_options.intra_op_num_threads=1; sess_options.inter_op_num_threads=1
sess = ort.InferenceSession(str(ONNX_PATH), sess_options=sess_options, providers=["CPUExecutionProvider"])

def validate_examples(examples, full_background=False):
    ok=0; bad=[]; outside_zero_ok=0
    for i,ex in enumerate(examples):
        x=grid_to_tensor(ex["input"], full_background=full_background)
        y=sess.run(None,{"input":x})[0]
        pred=(y>0.5).astype(np.float32)
        exp=grid_to_tensor(ex["output"], full_background=False)
        if np.array_equal(pred,exp): ok += 1
        else: bad.append(i)
        active=(exp.sum(axis=1,keepdims=True)>0.5).astype(np.float32)
        outside_zero_ok += bool(np.all(pred*(1-active)==0))
    return {"ok":ok,"total":len(examples),"bad_first10":bad[:10],"outside_zero_ok":outside_zero_ok}

rng=random.Random(0)
inds=list(range(len(task.get("arc-gen",[]))))
rng.shuffle(inds)
hold=[task["arc-gen"][i] for i in inds[:max(1, math.ceil(0.6*len(inds)))]] if inds else []
validation={
    "strict_zero_padding": {
        "train": validate_examples(task["train"], False),
        "test": validate_examples(task["test"], False),
        "arc_gen_60pct_holdout": validate_examples(hold, False) if hold else None,
    },
    "full_background_padding_diagnostic": {
        "train": validate_examples(task["train"], True),
        "test": validate_examples(task["test"], True),
    }
}
summary={"task_id":TASK_ID,"model_class":model.__class__.__name__,**summary_static,"validation":validation}
json.dump(summary, open(SUMMARY_PATH,"w"), indent=2)
print(json.dumps(summary, indent=2)[:3000])
assert validation["strict_zero_padding"]["test"]["ok"] == validation["strict_zero_padding"]["test"]["total"]
assert validation["strict_zero_padding"]["train"]["ok"] == validation["strict_zero_padding"]["train"]["total"]

{
  "task_id": "task328",
  "model_class": "Task328HybridCorners",
  "input_shape": [
    1,
    10,
    30,
    30
  ],
  "output_shape": [
    1,
    10,
    30,
    30
  ],
  "onnx_size_bytes": 96441,
  "ops": {
    "Constant": 269,
    "ReduceSum": 14,
    "Greater": 22,
    "Cast": 40,
    "Slice": 45,
    "Mul": 92,
    "ReduceMax": 4,
    "Min": 5,
    "Where": 3,
    "LessOrEqual": 10,
    "Equal": 11,
    "Sub": 7,
    "Add": 50,
    "Floor": 4,
    "Clip": 23,
    "Div": 2,
    "Concat": 1
  },
  "forbidden_ops": [],
  "validation": {
    "strict_zero_padding": {
      "train": {
        "ok": 4,
        "total": 4,
        "bad_first10": [],
        "outside_zero_ok": 4
      },
      "test": {
        "ok": 1,
        "total": 1,
        "bad_first10": [],
        "outside_zero_ok": 1
      },
      "arc_gen_60pct_holdout": {
        "ok": 158,
        "total": 158,
        "bad_first10": [],
        "outside_zero_ok": 158
      }
    },
    "full_background_padding_diagnos

In [11]:
with zipfile.ZipFile(SUBMISSION_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(ONNX_PATH, arcname=f"{TASK_ID}.onnx")
print("Wrote:", SUBMISSION_PATH)
print("Zip contents:", zipfile.ZipFile(SUBMISSION_PATH).namelist())
assert zipfile.ZipFile(SUBMISSION_PATH).namelist() == [f"{TASK_ID}.onnx"]

Wrote: /kaggle/working/submission.zip
Zip contents: ['task328.onnx']
